<h2>Food (v0.03) - 'Food (v0.03).ipynb'</h2>

<u><h5>Context</h5></u>

The notebook aims to isolate NZF impacts modelled for individual commodities with 'CPC' codes forwarded over by David Forengie on 23rd September 2026.

The objectives of the notebook are:

<ol>
    <li>
        To develop the representation of CPC codes provided by David, as its not completely clear how these work (e.g. where some CPC codes are only 3 digits, whereas others are 5). 
    </li>
    <li>
        To read-in 'v0.64' of the 'Impact' analysis results for the country in question, then isolate these records for the CPC codes identified in the step above.
    </li>
</ol>

<u><h5>Version Documentation</h5></u>

'v0.01' outlined an initial method for completing the process. 'v0.02' refined the method and developed a fully functioning notebook, utilising results from the 'Impact' notebooks which now include CPC codes as fields in the output data tables. Upon inspection, it was found that the identification of CPC codes wasn't functioning properly, where CPC codes specified by David wouldn't show in the dataset, yet HSa4 codes given in the CPC manual would show. This likely explains much of the losses observed when pulling by CPC codes, so 'v0.03' utilises a new version of the codes sent through David which have been mapped to the HS system ahead of time. 

In [1]:
# Import packages
import os
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def download_link(df, title="Download CSV file", filename="data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.03"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/food/"

# 1. Set up the notebook

## 1.1 Let's set the country we're interested in and set up a results folder.

In [2]:
# Select one of our countries to work on...
iso_code = "052"
iso_country = "Barbados"
year = 2025

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

## 1.2 Import and Format 'Impact Analysis' Results

In [3]:
impact_im_c = [
    "trade_idx", "reporterCode", "reporterDesc", "partnerCode", "partnerDesc", "flowDesc", "motDesc", 
    "HS6", "HS6_desc", "HS4", "HS4_desc", "HS2", "HS2_desc", "cpc", "cpc_desc", "eora_desc", "gtap", "gtap_desc", 
    "cifvalue", "fobvalue", "primaryValue", "qtyUnitAbbr", "qty", "netWgt", "vol_kg_vtw", 
    "sea_dist_km", "sea_dist_nm", "tnm", "fac_vessel_types", "fac_vessel_types_str", 
    "TandT_match_desc", "TandT_ad_val_pct", "TandT_N", "TandT_ciffob", "TandT_cifvalue", "TandT_fobvalue", "cost_int_match_desc",
    "bau_usd_26", "levy_usd_26", "asis_usd_26", "brazil_usd_26", "liscr_usd_26", "direct_usd_26", 
    "bau_usd_29", "levy_usd_29", "asis_usd_29", "brazil_usd_29", "liscr_usd_29", "direct_usd_29", 
    "bau_usd_32", "levy_usd_32", "asis_usd_32", "brazil_usd_32", "liscr_usd_32", "direct_usd_32", 
    "bau_usd_35", "levy_usd_35", "asis_usd_35", "brazil_usd_35", "liscr_usd_35", "direct_usd_35", 
    "bau_usd_38", "levy_usd_38", "asis_usd_38", "brazil_usd_38", "liscr_usd_38", "direct_usd_38", 
    "bau_usd_41", "levy_usd_41", "asis_usd_41", "brazil_usd_41", "liscr_usd_41", "direct_usd_41", 
    "bau_usd_44", "levy_usd_44", "asis_usd_44", "brazil_usd_44", "liscr_usd_44", "direct_usd_44", 
    "bau_usd_47", "levy_usd_47", "asis_usd_47", "brazil_usd_47", "liscr_usd_47", "direct_usd_47", 
    "bau_usd_50", "levy_usd_50", "asis_usd_50", "brazil_usd_50", "liscr_usd_50", "direct_usd_50"	
]

impact_im_d = {
    "reporterCode":"str", "partnerCode":"str", 
    "HS6":"str", "HS4":"str", "HS2":"str", "HS2":"str", "gtap":"str", "cpc":"str"
}

impact_im = pd.read_csv(
    input_dir + "impact/{0}/impact_im_v0.64.csv".format(iso_country), 
    dtype=impact_im_d,
    usecols=impact_im_c
)[impact_im_c]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(impact_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(impact_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(impact_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(impact_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(impact_im.vol_kg_vtw.sum() / 1e6, 1)))

impact_im["cpc_cl"] = [str(i).split('.')[0] for i in impact_im.cpc.values]
impact_im = impact_im.drop(columns="trade_idx").reset_index(drop=True).reset_index().rename(columns={"index":"trade_idx"})
display(impact_im.head(2))

Aggregate trade portfolio statistics for 'Barbados': 

Number of trade records present in dataset: 24503
Number of partner countries present in dataset: 135
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$1941.7m
Total weight within the dataset: 1956.2 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl
0,0,052,Barbados,008,Albania,Import,Sea,853710,"853710 | Boards, panels, cons",8537,"8537 | Boards, panels, cons",85,85 | Electrical machinery,46213.0,"Boards, consoles, cabinets and other bases, eq...",Electrical and Machinery,41.0,OME - Machinery and equipment n.e.c.,76062.73725,NaN,76062.73725,kg,2535.6645,2535.6645,1217.003796,7757.0,4819.974847,5865.927685,"['Container', 'General cargo']","['Container', 'General cargo']",3 - Reporter and HS4 code,19.52,74,12422.56,76062.74,63640.18,1. Primary Int. Inv. with Vess Type,34.328419,34.274138,34.279431,34.279431,34.303759,34.303759,35.764364,61.100586,38.855880,36.425118,36.417407,37.431742,35.085121,59.176607,40.652803,37.807706,37.012928,38.510792,34.489393,64.783312,49.852274,44.302627,41.339851,39.991020,33.904701,69.144290,56.804862,53.591455,43.679346,42.620613,33.331045,69.633501,66.855996,67.732491,47.068707,44.587883,32.745891,68.251131,65.242876,70.558471,53.799737,45.330617,32.194634,61.438164,63.520755,66.873782,51.297568,45.816048,31.632527,56.858079,57.582181,69.633191,57.748672,46.097076,46213
1,1,052,Barbados,020,Andorra,Import,Sea,640590,640590 | Footwear; n.e.c. in,6405,6405 | Footwear; other foot,64,64 | Footwear; gaiters an,2952.0,"Wooden footwear, miscellaneous special footwea...",Textiles and Wearing Apparel,29.0,LEA - Leather products,124.64320,NaN,124.64320,2u,35.4100,70.8200,8.450809,6652.0,4133.359892,34.930235,['Container'],['Container'],3 - Reporter and HS4 code,8.63,22,9.90,124.64,114.74,1. Primary Int. Inv. with Vess Type,0.182063,0.181745,0.181776,0.181776,0.181919,0.181919,0.190452,0.337591,0.210037,0.195040,0.195570,0.200461,0.186484,0.330597,0.221100,0.202538,0.197886,0.206854,0.183004,0.359569,0.274191,0.239803,0.221857,0.215606,0.179588,0.382289,0.314574,0.294739,0.237521,0.231079,0.176237,0.387445,0.370218,0.373430,0.257460,0.242743,0.172819,0.379478,0.363583,0.398323,0.294775,0.247193,0.169599,0.338451,0.352799,0.369681,0.277012,0.250114,0.166315,0.310901,0.316022,0.393937,0.325504,0.251772,2952


# 2. Subset the 'Impact Analysis' results for food security-related CPC codes

## 2.1 Let's get a sense of how much of import value is associated with food-relevant goods across classification systems

### 2.1.1 Harmonised System (HS01-23)

In [5]:
#print("HS groupings: \n\n{0}\n".format(pd.Series(impact_im.HS2_desc.unique()).sort_values().values))
HSa2_good_groups = [
    "01", "02", "03", "04", "05", "07", "08", # "06", 
    "09", "10", "11", "12", "13", "14", "15", "16", 
    "17", "18", "19", "20", "21", "22", "23"
]

impact_im_HSa2_food = impact_im[(impact_im.HS2.isin(HSa2_good_groups))]
impact_im_HSa2_food_g = impact_im_HSa2_food.groupby(["HS2_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_HSa2_food_g["pct_val"] = round(100 * impact_im_HSa2_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_HSa2_food_g["pct_vol"] = round(100 * impact_im_HSa2_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_HSa2_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_HSa2_food_g.round(1)
))

Value of food imports identified via HSa2 ('Fishing' and 'Food & Beverages'): $454.0m (23.4%), ... 

                             cifvalue   vol_kg_vtw  pct_val  pct_vol
HS2_desc                                                            
22 | Beverages, spirits a  56978418.2  410079791.2     12.6     56.0
04 | Dairy produce; birds  49273080.0   18344140.8     10.9      2.5
21 | Miscellaneous edible  45662514.0   13287978.9     10.1      1.8
19 | Preparations of cere  37338799.7   14453507.9      8.2      2.0
20 | Preparations of vege  31888686.7   20766219.9      7.0      2.8
02 | Meat and edible meat  31676973.9    9134508.1      7.0      1.2
08 | Fruit and nuts, edib  31499953.6   23919084.9      6.9      3.3
03 | Fish and crustaceans  27426835.2    7243691.1      6.0      1.0
16 | Meat, fish, crustace  21891496.5    4669443.0      4.8      0.6
07 | Vegetables and certa  20731429.5   29320881.7      4.6      4.0
17 | Sugars and sugar con  20310855.9   64149232.0      4.5      8.8
23

### 2.1.2 EORA groupings ('Food & Beverages' and 'Fishing')

In [14]:
print("EORA groupings: \n\n{0}\n".format(pd.Series(impact_im.eora_desc.unique()).sort_values().values))
eora_good_groups = ["Food & Beverages", "Agriculture", "Fishing"]

impact_im_eora_food = impact_im[(impact_im.eora_desc.isin(eora_good_groups))]
impact_im_eora_food_g = impact_im_eora_food.groupby(["eora_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_eora_food_g["pct_val"] = round(100 * impact_im_eora_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_eora_food_g["pct_vol"] = round(100 * impact_im_eora_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_eora_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_eora_food_g.round(1)
))

EORA groupings: 

['Agriculture' 'Electrical and Machinery' 'Fishing' 'Food & Beverages'
 'Metal Products' 'Mining and Quarrying' 'Other Manufacturing'
 'Petroleum, Chemical and Non-Metallic Mineral Products'
 'Textiles and Wearing Apparel' 'Transport Equipment' 'Wood and Paper' nan]

Value of food imports identified via EORA ('Fishing' and 'Food & Beverages'): $403.6m (20.8%), ... 

                     cifvalue   vol_kg_vtw  pct_val  pct_vol
eora_desc                                                   
Food & Beverages  234803105.8  539189499.8     12.1     27.6
Agriculture       156233569.2  134036865.6      8.0      6.9
Fishing            12525699.7    1881261.0      0.6      0.1



### 2.1.3 GTAP (16x headings)

In [7]:
#print("GTAP groupings: \n\n{0}\n".format(pd.Series(impact_im.gtap_desc.unique()).sort_values().values))
gtap_food_groups = [
    "GRO - Cereal grains n.e.c.", "PDR - Paddy rice", "PCR - Processed rice", "SGR - Sugar", 
    "CMT - Bovine meat prods", "CTL - Bovine cattle, sheep and goats, horses", "FSH - Fishing", "OMT - Meat products n.e.c.",
    "MIL - Dairy products", "OFD - Food products n.e.c.", "V_F - Vegetables, fruit, nuts", "OSD - Oil seeds", 
    "OCR - Crops n.e.c.", "PFB - Plant-based fibers", "VOL - Vegetable oils and fats", 
    "OAP - Animal products n.e.c."
]

impact_im_gtap_food = impact_im[(impact_im.gtap_desc.isin(gtap_food_groups))]
impact_im_gtap_food_g = impact_im_gtap_food.groupby(["gtap_desc"]).agg({"cifvalue":"sum", "vol_kg_vtw":"sum"}).sort_values(by="cifvalue", ascending=False)
impact_im_gtap_food_g["pct_val"] = round(100 * impact_im_gtap_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_gtap_food_g["pct_vol"] = round(100 * impact_im_gtap_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Value of food imports identified via GTAP: ${0}m ({1}%), ... \n\n{2}\n".format(
    round(impact_im_gtap_food.cifvalue.sum() / 1e6, 1), 
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    impact_im_gtap_food_g.head(15).round(1)
))

Value of food imports identified via GTAP: $348.4m (17.9%), ... 

                                                 cifvalue  vol_kg_vtw  \
gtap_desc                                                               
OFD - Food products n.e.c.                    180454532.5  91364832.7   
V_F - Vegetables, fruit, nuts                  45515972.0  47336076.5   
MIL - Dairy products                           35744920.8  10243320.8   
OMT - Meat products n.e.c.                     23089747.2   7510480.2   
CMT - Bovine meat prods                        18787813.7   3860726.8   
SGR - Sugar                                    14824493.2  60341815.8   
VOL - Vegetable oils and fats                  14383354.8  14537204.5   
GRO - Cereal grains n.e.c.                      5629970.1  23237086.9   
OCR - Crops n.e.c.                              4705369.4   1667372.9   
PCR - Processed rice                            2314157.8   4019692.2   
PDR - Paddy rice                                1748404.7 

## 2.2 Comparison of results

In [8]:
print("Share of total import value associated with food-relevant goods by classification system...\n\nHSa2 {0}% \nGTAP {1}% \nEORA {2}%".format(
    round(100 * impact_im_HSa2_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_gtap_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1),
    round(100 * impact_im_eora_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))

Share of total import value associated with food-relevant goods by classification system...

HSa2 23.4% 
GTAP 17.9% 
EORA 20.8%


# 3. Take HSa4 codes associated with specific CPCs

## 3.1 Set HSa4 Codes we're interested in

In [9]:
hsa4_codes_food = {
    "Dairy": [
        "0401", "0402", "0403", "0404", "0405", "0406",
    ],
    "Fats and Oils": [
        "1501", "1502", "1503", "1504", "1505", "1506", "1507", "1508", "1509", "1510"
        "1511", "1512", "1513", "1514", "1515", "1516", "1517"
    ],
    "Fruits": [
        "0709", "0801", "0802", "0803", "0804", "0805", "0806", "0807", "0808", "0809", 
        "0810", "0813", "2009"
    ],
    "Meat and Prep": [
        "0201", "0203", "0204", "0205", "0207", "0208", "0209", "2301"
    ],
    "Processed Cereals": [
        "1006", "1101", "1102", "1104", "1105", "1106", "1107", "1108", "1109", "1302",
        "1602", "1901", "1902", "1903", "1904", "1905", "2005", "2007", "2104", "2106", 
        "3505"
    ],
    "Sugar and Honey": [
        "1701", "1702", "1703", "1704", "2006"
    ],
    "Unprocessed Cereals": [
        "1001", "1002", "1003", "1004", "1005", "1007", "1008"
    ],
    "Vegetables": [
        "0701", "0702", "0703", "0704", "0705", "0706", "0707", "0708", "0709", "0710",
        "0712", "0713", "0714", "1209", "2009"
    ]
}

## 3.2 Identify 'Food Security' groups of interest via CPC

In [10]:
impact_im["food_group"] = np.nan

for food_group in list(hsa4_codes_food.keys()):
    impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group

display(impact_im.head(2))

/var/folders/_k/81z8ryjd28lc4f0bb4dm7gkw0000gn/T/ipykernel_21382/1412186970.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Dairy' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  impact_im.loc[impact_im.HS4.isin(hsa4_codes_food[food_group]), "food_group"] = food_group


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,flowDesc,motDesc,HS6,HS6_desc,HS4,HS4_desc,HS2,HS2_desc,cpc,cpc_desc,eora_desc,gtap,gtap_desc,cifvalue,fobvalue,primaryValue,qtyUnitAbbr,qty,netWgt,vol_kg_vtw,sea_dist_km,sea_dist_nm,tnm,fac_vessel_types,fac_vessel_types_str,TandT_match_desc,TandT_ad_val_pct,TandT_N,TandT_ciffob,TandT_cifvalue,TandT_fobvalue,cost_int_match_desc,bau_usd_26,levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,direct_usd_26,bau_usd_29,levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,direct_usd_29,bau_usd_32,levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,direct_usd_32,bau_usd_35,levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,direct_usd_35,bau_usd_38,levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,direct_usd_38,bau_usd_41,levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,direct_usd_41,bau_usd_44,levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,direct_usd_44,bau_usd_47,levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,direct_usd_47,bau_usd_50,levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50,direct_usd_50,cpc_cl,food_group
0,0,052,Barbados,008,Albania,Import,Sea,853710,"853710 | Boards, panels, cons",8537,"8537 | Boards, panels, cons",85,85 | Electrical machinery,46213.0,"Boards, consoles, cabinets and other bases, eq...",Electrical and Machinery,41.0,OME - Machinery and equipment n.e.c.,76062.73725,NaN,76062.73725,kg,2535.6645,2535.6645,1217.003796,7757.0,4819.974847,5865.927685,"['Container', 'General cargo']","['Container', 'General cargo']",3 - Reporter and HS4 code,19.52,74,12422.56,76062.74,63640.18,1. Primary Int. Inv. with Vess Type,34.328419,34.274138,34.279431,34.279431,34.303759,34.303759,35.764364,61.100586,38.855880,36.425118,36.417407,37.431742,35.085121,59.176607,40.652803,37.807706,37.012928,38.510792,34.489393,64.783312,49.852274,44.302627,41.339851,39.991020,33.904701,69.144290,56.804862,53.591455,43.679346,42.620613,33.331045,69.633501,66.855996,67.732491,47.068707,44.587883,32.745891,68.251131,65.242876,70.558471,53.799737,45.330617,32.194634,61.438164,63.520755,66.873782,51.297568,45.816048,31.632527,56.858079,57.582181,69.633191,57.748672,46.097076,46213,NaN
1,1,052,Barbados,020,Andorra,Import,Sea,640590,640590 | Footwear; n.e.c. in,6405,6405 | Footwear; other foot,64,64 | Footwear; gaiters an,2952.0,"Wooden footwear, miscellaneous special footwea...",Textiles and Wearing Apparel,29.0,LEA - Leather products,124.64320,NaN,124.64320,2u,35.4100,70.8200,8.450809,6652.0,4133.359892,34.930235,['Container'],['Container'],3 - Reporter and HS4 code,8.63,22,9.90,124.64,114.74,1. Primary Int. Inv. with Vess Type,0.182063,0.181745,0.181776,0.181776,0.181919,0.181919,0.190452,0.337591,0.210037,0.195040,0.195570,0.200461,0.186484,0.330597,0.221100,0.202538,0.197886,0.206854,0.183004,0.359569,0.274191,0.239803,0.221857,0.215606,0.179588,0.382289,0.314574,0.294739,0.237521,0.231079,0.176237,0.387445,0.370218,0.373430,0.257460,0.242743,0.172819,0.379478,0.363583,0.398323,0.294775,0.247193,0.169599,0.338451,0.352799,0.369681,0.277012,0.250114,0.166315,0.310901,0.316022,0.393937,0.325504,0.251772,2952,NaN


## 3.3 Inspect the output

In [11]:
impact_im_cpc_food = impact_im[(~impact_im.food_group.isna())]
impact_im_cpc_food_g = impact_im_cpc_food.groupby("food_group").agg({"HS6_desc":"count", "cifvalue":"sum", "vol_kg_vtw":"sum"}).reset_index().sort_values(by="cifvalue", ascending=False).round(2).rename(columns={"HS6_desc":"N_com"}).reset_index(drop=True)
impact_im_cpc_food_g["pct_val"] = round(100 * impact_im_cpc_food_g.cifvalue / impact_im.cifvalue.sum(), 1)
impact_im_cpc_food_g["pct_vol"] = round(100 * impact_im_cpc_food_g.vol_kg_vtw / impact_im.vol_kg_vtw.sum(), 1)

print("Proportion of imports identified as food security-relevant:\n")
print("Value: ${0}m of ${1}m ({2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / 1e6, 1),
    round(impact_im.cifvalue.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.cifvalue.sum() / impact_im.cifvalue.sum(), 1)
))
print("Volume: {0} of {1} million kilograms ({2}%)".format(
    round(impact_im_cpc_food.vol_kg_vtw.sum() / 1e6, 1),
    round(impact_im.vol_kg_vtw.sum() / 1e6, 1),
    round(100 * impact_im_cpc_food.vol_kg_vtw.sum() / impact_im.vol_kg_vtw.sum(), 1)
))
print("Value-to-Weight: {0} from {1} US$ per kg (+{2}%)".format(
    round(impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum(), 2),
    round(impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum(), 2),
    round(100 * ((impact_im_cpc_food.cifvalue.sum() / impact_im_cpc_food.vol_kg_vtw.sum()) - (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum())) / (impact_im.cifvalue.sum() / impact_im.vol_kg_vtw.sum()), 1)
))
print("Average distance to market (volume-weighted): {0} from {1} nm (+{2}%)".format(
    round((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000), 1),
    round((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000), 1),
    round(100 * (((impact_im_cpc_food.vol_kg_vtw / 1000 * impact_im_cpc_food.sea_dist_nm).sum() / (impact_im_cpc_food.vol_kg_vtw.sum() / 1000)) - ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000))) / \
          ((impact_im.vol_kg_vtw / 1000 * impact_im.sea_dist_nm).sum() / (impact_im.vol_kg_vtw.sum() / 1000)), 1)
))
print("\n{0}\n".format(impact_im_cpc_food_g.head(15).round(1)))

impact_im_cpc_food_g.to_csv(output_dir + "{0}/food_total_usd_{1}.csv".format(iso_country, version), index=False)

Proportion of imports identified as food security-relevant:

Value: $249.6m of $1941.7m (12.9%)
Volume: 244.6 of 1956.2 million kilograms (12.5%)
Value-to-Weight: 1.02 from 0.99 US$ per kg (+2.8%)
Average distance to market (volume-weighted): 2482.7 from 2222.4 nm (+11.7%)

            food_group  N_com    cifvalue  vol_kg_vtw  pct_val  pct_vol
0    Processed Cereals    475  93880026.3  55586925.2      4.8      2.8
1                Dairy     96  37264071.7  11918770.5      1.9      0.6
2               Fruits    200  30125158.1  23272052.2      1.6      1.2
3           Vegetables    346  29583625.7  36466624.6      1.5      1.9
4      Sugar and Honey     95  20582385.0  64227079.4      1.1      3.3
5        Meat and Prep     54  17272925.9   5900522.6      0.9      0.3
6  Unprocessed Cereals     29  10431247.4  40524031.6      0.5      2.1
7        Fats and Oils     95  10430598.7   6654369.0      0.5      0.3



## 3.3 Derive results for 2029, 2041 and 2050

### 3.3.1 Total FEM costs by scenario (US$)

In [16]:
print("\nTotal FEM costs by scenario (US$)...\n")

d = {"food_group": list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["bau", "levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            round(100 * impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum(), 1) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$)...



,food_group,bau_29,levy_29,asis_29,brazil_29,liscr_29,direct_29,bau_41,levy_41,asis_41,brazil_41,liscr_41,direct_41,bau_50,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,23418058.8,41232401.8,25750615.5,23946661.4,24007039.0,24621499.9,21697647.2,47295856.8,45174832.5,45630395.6,31505965.1,29732189.7,20495763.4,38030932.1,38678834.3,47982082.7,39633997.9,30833608.3
1,Fats and Oils,12896993.0,22651828.8,14173526.9,13177122.8,13212853.5,13556492.0,11952928.2,26005221.5,24804213.1,25055445.3,17350521.2,16359235.5,11293757.6,20890648.4,21294627.3,26384228.8,21789581.7,16957588.9
2,Fruits,28212733.0,49872834.9,31076923.3,28875014.8,28951158.4,29682080.0,26120508.9,57224773.7,54671953.8,55177736.8,38064777.1,35901992.7,24659639.2,45958661.6,46725674.3,58131357.8,48026815.8,37235125.3
3,Meat and Prep,11357606.7,20132274.3,12525604.2,11631235.8,11662862.8,11954518.3,10509920.7,23105326.8,22077973.5,22269548.7,15353680.9,14475991.2,9918214.2,18540617.0,18846015.5,23492490.0,19411458.8,15014452.1
4,Processed Cereals,51893853.3,91033883.1,56974441.6,53022910.8,53152236.4,54528793.0,48114142.9,104432877.5,99737237.8,100823150.9,69669123.5,65745784.1,45473011.2,84063731.3,85510488.3,105796908.0,87398208.0,68170580.3
5,Sugar and Honey,100402963.6,173912637.2,109775553.9,102229626.0,102504769.7,105326458.1,93276614.7,199327891.3,189719057.4,192126445.9,134078421.2,126390664.8,88301770.7,160869035.7,164507847.6,202443526.3,167033764.3,130879459.0
6,Unprocessed Cereals,46803113.9,82160714.9,51424896.8,47829870.9,47960358.3,49193529.2,43384888.2,94583737.5,90369873.2,91347699.7,63120630.7,59327756.1,40998812.2,76036848.0,77300428.4,95719554.7,79233843.9,61482304.7
7,Vegetables,55681332.7,97878715.2,61180684.9,56915824.2,57057122.1,58525966.8,51607149.0,112280252.8,107233416.2,108358204.8,74826417.9,70626223.7,48758614.2,90318253.0,91866685.4,113803263.4,93998044.9,73237622.2


### 3.3.2 Total FEM costs by scenario (% cif)

In [15]:
print("\nTotal FEM costs by scenario (% cif)...\n")

d = {"food_group":list(hsa4_codes_food.keys())}
for year in ["29", "41", "50"]:
    for scenario in ["levy", "asis", "brazil", "liscr", "direct"]:
        d_temp = {
            "{0}_{1}".format(scenario, year): [
            np.round(100 * (impact_im[(impact_im.food_group == hsa4_codes_food_key)]["{0}_usd_{1}".format(scenario, year)].sum() - impact_im[(impact_im.food_group == hsa4_codes_food_key)]["bau_usd_{1}".format(scenario, year)].sum()) / \
                     impact_im[(impact_im.food_group == hsa4_codes_food_key)].cifvalue.sum(), 2) \
                    for hsa4_codes_food_key in list(hsa4_codes_food.keys())]}
        d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/food_total_costs_pct_cif_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (% cif)...



,food_group,levy_29,asis_29,brazil_29,liscr_29,direct_29,levy_41,asis_41,brazil_41,liscr_41,direct_41,levy_50,asis_50,brazil_50,liscr_50,direct_50
0,Dairy,0.48,0.06,0.01,0.02,0.03,0.69,0.63,0.64,0.26,0.22,0.47,0.49,0.74,0.51,0.28
1,Fats and Oils,0.94,0.12,0.03,0.03,0.06,1.35,1.23,1.26,0.52,0.42,0.92,0.96,1.45,1.01,0.54
2,Fruits,0.72,0.10,0.02,0.02,0.05,1.03,0.95,0.96,0.40,0.32,0.71,0.73,1.11,0.78,0.42
3,Meat and Prep,0.51,0.07,0.02,0.02,0.03,0.73,0.67,0.68,0.28,0.23,0.50,0.52,0.79,0.55,0.30
4,Processed Cereals,0.42,0.05,0.01,0.01,0.03,0.60,0.55,0.56,0.23,0.19,0.41,0.43,0.64,0.45,0.24
5,Sugar and Honey,3.57,0.46,0.09,0.10,0.24,5.15,4.69,4.80,1.98,1.61,3.53,3.70,5.55,3.83,2.07
6,Unprocessed Cereals,3.39,0.44,0.10,0.11,0.23,4.91,4.50,4.60,1.89,1.53,3.36,3.48,5.25,3.67,1.96
7,Vegetables,1.43,0.19,0.04,0.05,0.10,2.05,1.88,1.92,0.78,0.64,1.40,1.46,2.20,1.53,0.83
